**Calculate Correlation Coefficient between station observations and incident pixels**


In [67]:
import xarray
import pandas as pd
import glob
import os
import rasterio
import concurrent.futures
from scipy.stats import spearmanr
import os
import numpy as np
from permetrics.regression import RegressionMetric
import HydroErr as he
import hydroeval as hel

In [68]:
datasets = ['CHIRPS', 'PERSIANN', 'ERA5L'] 

#Read insitu data
df_insitu = pd.read_csv(r'..\Data\Processed\Insitu\insitu_rainfall_1990_2019.csv')
df_insitu['date'] = pd.to_datetime(df_insitu[['year', 'month', 'day']])

#Get path, files and file list for datasets
def extract_date(file, dataset):
    filename = os.path.basename(file)
    date_str = filename[-12:-4]
    return date_str

def get_dataset_info(dataset):
    dataset_path = fr"..\Data\Datasets\{dataset}_prec"
    dataset_tiff_files = glob.glob(os.path.join(dataset_path, "*.tif"))
    dataset_dates = list(map(lambda file: extract_date(file, dataset), dataset_tiff_files))
    return dataset_path, dataset_tiff_files, dataset_dates

dataset_results = {}
for dataset in datasets:
    dataset_path, dataset_tiff_files, dataset_dates = get_dataset_info(dataset)
    dataset_results[f'{dataset}_path'] = dataset_path
    dataset_results[f'{dataset}_files'] = dataset_tiff_files
    dataset_results[f'{dataset}_dates'] = dataset_dates

#dataset_results['CHIRPS_files']

In [69]:
#extract station lat and lon in a list 
unique_stn = df_insitu[['index_no','lat','long']].drop_duplicates()
stn_points = list(zip(unique_stn['long'],unique_stn['lat']))
#stn_points
#unique_stn   

In [70]:
def extract_concurrent_grid_data(dataset_results, stn_points, unique_stn):
    df_final = None
    
    for dataset in datasets:
        files = dataset_results[f'{dataset}_files']
        print(f"Processing {dataset} with {len(files)} files...")
        
        def process_file(file):
            date_obj = extract_date(file, dataset) 
            local_records = []
            try:
                with rasterio.open(file) as src:
                    crs = src.crs
                    if crs == "EPSG:4326":
                        sampled_values = list(src.sample(stn_points))
                        for i, value in enumerate(sampled_values):
                            local_records.append({
                                'date': date_obj,
                                'stn_id': unique_stn.iloc[i]['index_no'],
                                'prec_grid': value[0]
                                })
            except Exception as e:
                print(f'Error processing {file}: {e}')
            return local_records

        dataset_records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            results = executor.map(process_file, files)
            for result in results:
                dataset_records.extend(result)
        
        df_dataset = pd.DataFrame(dataset_records)
        
        # CHECK IF EMPTY FIRST
        if not df_dataset.empty:
            df_dataset = df_dataset[df_dataset['prec_grid'] != -9999.0]
            df_dataset = df_dataset.dropna().round(1)
            df_dataset.loc[df_dataset['prec_grid'] < 0, 'prec_grid'] = 0
            df_dataset = df_dataset.rename(columns={'prec_grid': f'{dataset}_prec'})
            
            # Merge logic
            if df_final is None:
                df_final = df_dataset
            else:
                df_final = pd.merge(df_final, df_dataset, on=['date', 'stn_id'], how='inner')
                print(f"Merged {dataset}. Resulting shape: {df_final.shape}")
        else:
            print(f"Warning: No data extracted for {dataset}")

    return df_final

df_grid = extract_concurrent_grid_data(dataset_results, stn_points, unique_stn)

Processing CHIRPS with 10957 files...
Processing PERSIANN with 10957 files...
Merged PERSIANN. Resulting shape: (936282, 4)
Processing ERA5L with 10956 files...
Merged ERA5L. Resulting shape: (936196, 5)


In [71]:
#merging station and grid df into a single df
df_insitu1 = df_insitu[['date','index_no','rainfall_sum']]
df_insitu1 = df_insitu1.rename(columns={'rainfall_sum':'prec_insitu', 'index_no':'stn_id'})

df_grid['date'] = pd.to_datetime(df_grid['date'])
df_insitu1['date'] = pd.to_datetime(df_insitu1['date'])
df_merged = pd.merge(df_grid, df_insitu1, on=['date','stn_id'], how='inner')
df_merged

,date,stn_id,CHIRPS_prec,PERSIANN_prec,ERA5L_prec,prec_insitu
0,1990-01-01,102.0,0.0,0.0,0.0,0.0
1,1990-01-01,103.0,0.0,0.0,0.0,0.0
2,1990-01-01,104.0,0.0,0.0,0.0,0.0
3,1990-01-01,105.0,0.0,0.0,0.0,0.0
4,1990-01-01,107.0,0.0,0.2,0.0,0.0
...,...,...,...,...,...,...
900077,2019-12-31,1407.0,0.0,0.0,0.1,0.0
900078,2019-12-31,1408.0,0.0,0.0,0.0,0.0
900079,2019-12-31,1412.0,0.0,0.0,0.0,0.0
900080,2019-12-31,1416.0,0.0,0.0,0.0,0.0


**Calculating error metrices - Example ERA5L**

In [72]:
df_accuracy = pd.DataFrame()
for dataset in datasets:
    dataset_accuracy = {'dataset': dataset}

    #Convert to array
    prec_grid_array = df_merged[f'{dataset}_prec'].values
    prec_insitu_array = df_merged['prec_insitu'].values
    
    #drop null values
    valid_indices = np.logical_and(~np.isnan(prec_grid_array), ~np.isnan(prec_insitu_array))
    observations = len(valid_indices)
    dataset_accuracy['observations_n'] = observations

    prec_grid_array = prec_grid_array[valid_indices]
    prec_insitu_array = prec_insitu_array[valid_indices]

    #correlation coefficient
    correlation = he.spearman_r(prec_grid_array, prec_insitu_array)
    dataset_accuracy['correlation'] = correlation

    #NRMSE
    nrmse = he.nrmse_mean(prec_grid_array, prec_insitu_array)
    dataset_accuracy['nrmse'] = nrmse

    #Percent bias
    pbias = 100 * np.sum(prec_grid_array - prec_insitu_array) / np.sum(prec_insitu_array)
    dataset_accuracy['pbias'] = pbias

    #Relative standard deviation 
    rsd = (np.std(prec_grid_array))/(np.std(prec_insitu_array))
    dataset_accuracy['rsd'] = rsd

    df_dataset_accuracy = pd.DataFrame([dataset_accuracy])
    df_accuracy = pd.concat([df_accuracy, df_dataset_accuracy], ignore_index=True)

df_accuracy


,dataset,observations_n,correlation,nrmse,pbias,rsd
0,CHIRPS,900082,0.279015,3.433587,-11.235871,0.936553
1,PERSIANN,900082,0.416261,2.944956,-29.541271,0.573728
2,ERA5L,900082,0.579364,3.240967,35.544650,1.033149


**Code for CPI Calculation after accuracy metrices are calculated for individual datasets**

In [ ]:
#Normalization
df_accuracy['abs_pbias'] = df_accuracy['pbias'].abs()
df_accuracy['diff_rsd'] = (df_accuracy['rsd']-1).abs()

def normalize_column(col):
    return (col - col.min()) / (col.max() - col.min())

df_accuracy_normalized = df_accuracy.copy().drop(columns=['pbias'])

metrics = ['correlation', 'nrmse', 'abs_pbias', 'diff_rsd']
for column in metrics:
    df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

#converting to scale of 0 to 1 (ideal to worst) - correlation & rsd needs inversion
df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

#get cpi for each (linear)
df_accuracy_normalized['cpi'] = df_accuracy_normalized['correlation'] + df_accuracy_normalized['nrmse'] + df_accuracy_normalized['abs_pbias'] + df_accuracy_normalized['diff_rsd']
df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(df_accuracy[['dataset','correlation','nrmse','pbias','rsd', 'observations_n']], on=['dataset'], how='inner')
df_accuracy_final

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,CHIRPS,2.077068,2.0,0.279015,3.433587,-11.235871,0.936553,900082
1,PERSIANN,2.296081,3.0,0.416261,2.944956,-29.541271,0.573728,900082
2,ERA5L,1.605798,1.0,0.579364,3.240967,35.544650,1.033149,900082
